# Self-Healing Agent — Pipeline Checkpoint Onboarding

This notebook **modifies Fabric pipeline definitions** to make them checkpoint-aware. After onboarding, each pipeline will:

1. **On start**: Load the list of previously-completed activities from a checkpoint Delta table
2. **Per activity**: Skip (via IfCondition) if already marked COMPLETED in the checkpoint table
3. **On activity success**: Write COMPLETED to the checkpoint table via a helper notebook
4. **On activity failure**: Write FAILED + error details to the checkpoint table, then re-raise the error
5. **On full success**: `_chk_reset` clears all checkpoint records (clean slate for next run)
6. **Retry = 0**: All business-activity retries are disabled — the self-healing agent performs RCA before deciding

### How it works

For each original activity `X`, the onboarding process wraps it:

```
Original:   A → B → C

Modified:   _chk_load → _chk_var
             ↓
            _chk_if_A (IfCondition — skip if COMPLETED)
              True-branch:
                A (retry=0)
                 ├─ on Success → _chk_upd_A (write COMPLETED)
                 └─ on Failure → _chk_fail_A (write FAILED + error)
                                   └→ _chk_re_fail_A (Fail activity — re-raise)
             ↓ (Succeeded)
            _chk_if_B ...
             ↓ (All Succeeded)
            _chk_reset (delete checkpoints — clean slate for next full run)
```

### Helper Notebook Modes

| Mode | Purpose |
|---|---|
| `CHECK_ALL` | Returns comma-separated list of COMPLETED activities for a pipeline. If any records exist → return completed list (so they get skipped). If no records → return empty (fresh run). |
| `UPDATE` | MERGE a checkpoint record (COMPLETED or FAILED) into the Delta table. On FAILED, optionally triggers PipelineMonitoring agent. |
| `RESET` | DELETE all checkpoint records for a pipeline. Called by `_chk_reset` after all activities succeed. |

### Retry Policy Override & Self-Healing RCA

When onboarding, the tool **sets retry = 0** on all business activities:

| Scenario | Without Onboarding | With Onboarding |
|---|---|---|
| Permanent failure (schema, auth, missing) | Retries N times, wastes time & compute | Fails once → error captured → agent fixes root cause |
| Bad data from upstream activity | Downstream retries with same bad input | Fails once → agent traces to upstream → can invalidate checkpoint |

**Self-healing agent workflow after failure:**
1. Pipeline fails → `_chk_fail_X` has written FAILED + full error to checkpoint table
2. Agent queries checkpoint: `SELECT * FROM checkpoint WHERE status = 'FAILED'`
3. Agent classifies failure → updates `failure_category` (TRANSIENT / PERMANENT / DATA_QUALITY / AUTH)
4. Agent updates `rca_notes` with findings
5. Agent takes action: fix root cause → reset the failed checkpoint → re-run pipeline
6. On re-run: previously COMPLETED activities are skipped, only the fixed activity re-executes

**Original retry policies** are preserved in the checkpoint table (`original_retry_count`) and in the backup table (full original definition).

### What gets deployed

1. **Checkpoint Delta table** in your Lakehouse — stores activity completion records
2. **`_checkpoint_helper` notebook** in your workspace — called by pipelines to check/update/reset checkpoints
3. **Modified pipeline definitions** — original activities wrapped with IfCondition + checkpoint logic

### Limitations & Edge Cases

- **Retry override**: Activities with retry policies will no longer auto-retry — this is intentional for RCA
- Original pipeline definitions are backed up to a Delta table before modification
- **"Skipped" dependency conditions**: If any activity uses `dependencyConditions: ["Skipped"]`, the checkpoint IfCondition may change behavior (it succeeds rather than "skips" when completed). Review these pipelines manually
- If the checkpoint helper notebook fails, the pipeline fails (checkpoint infra is a dependency)
- **Concurrent runs**: Checkpoint is keyed on `pipeline_name + activity_name` (not run_id). Run one instance at a time, or reset checkpoints between runs
- Inner activities (inside ForEach, Switch, etc.) are NOT individually checkpointed — the parent control activity is the checkpoint unit
- **SecureInput/SecureOutput**: Error details may be masked for activities with secure settings — the error object is still captured but content may be redacted
- Activity names containing commas are not supported
- Adds ~30-60s overhead per activity (helper notebook execution time)

## 1. Parameters

In [ ]:
# ── REQUIRED ──
WORKSPACE_ID         = ""       # Fabric workspace GUID
CHECKPOINT_LAKEHOUSE = ""       # Lakehouse name (e.g., "lh_pipeline_metadata")

# ── CHECKPOINT CONFIG ──
CHECKPOINT_TABLE       = "pipeline_activity_checkpoints"
HELPER_NOTEBOOK_NAME   = "_checkpoint_helper"
BACKUP_TABLE           = "pipeline_definition_backups"

# ── PIPELINE FILTERS ──
# Empty INCLUDE = onboard ALL pipelines. Supports fnmatch wildcards.
INCLUDE_PIPELINES = []            # e.g., ["Daily_*", "HR_Ingest"]
EXCLUDE_PIPELINES = ["_shadow*", "_checkpoint*"]   # Always excluded

# ── SAFETY ──
DRY_RUN = True   # True = preview only; False = apply changes

# ── AGENT NOTIFICATION — call PipelineMonitoring on failure ──
AGENT_WORKSPACE_ID       = ""   # Workspace GUID where PipelineMonitoring lives (e.g., HotFix Agent workspace)
AGENT_PIPELINE_NAME      = "PipelineMonitoring"  # Pipeline that calls the AI Foundry Agent
AGENT_PIPELINE_ID        = ""   # Auto-resolved below if empty; or set manually
NOTIFY_AGENT_ON_FAILURE  = True # True = helper notebook calls PipelineMonitoring on every FAILED activity

## 2. Initialize

In [ ]:
import json
import copy
import time
import re
import base64
import logging
import fnmatch
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

import sempy.fabric as fabric

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("onboard-checkpoint")

FABRIC_API = "https://api.fabric.microsoft.com/v1"

rest_client = fabric.FabricRestClient()
spark = SparkSession.builder.getOrCreate()


def _wait_for_long_operation(resp, rest_client, max_polls=60):
    """Handle Fabric REST API 202 long-running operations."""
    if resp.status_code == 202:
        location = resp.headers.get("Location")
        retry_after = int(resp.headers.get("Retry-After", "2"))
        for _ in range(max_polls):
            time.sleep(retry_after)
            resp = rest_client.get(location)
            if resp.status_code == 200:
                return resp
    return resp


print("✅ Fabric REST client initialized")
print("✅ Spark session ready")

## 3. Create Checkpoint Table & Backup Table

In [ ]:
assert CHECKPOINT_LAKEHOUSE, "Set CHECKPOINT_LAKEHOUSE in the Parameters cell"

# ── Checkpoint table: stores per-activity completion status ──
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE} (
        pipeline_name       STRING      NOT NULL,
        pipeline_id         STRING,
        activity_name       STRING      NOT NULL,
        activity_type       STRING,
        status              STRING      NOT NULL,
        run_id              STRING,
        error_message       STRING,
        resolved_at         TIMESTAMP,
        resolution_method   STRING,
        needs_rerun         BOOLEAN,
        original_retry_count STRING,
        failure_category    STRING,
        rca_notes           STRING
    )
    USING DELTA
""")
# ── Add columns for existing tables (safe upgrade) ──
for col_def in ["original_retry_count STRING", "failure_category STRING", "rca_notes STRING"]:
    try:
        spark.sql(f"ALTER TABLE {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE} ADD COLUMNS ({col_def})")
    except Exception:
        pass  # Column already exists
print(f"✅ Checkpoint table ready: {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE}")

# ── Backup table: stores original pipeline definitions for rollback ──
spark.sql(f"""

    CREATE TABLE IF NOT EXISTS {CHECKPOINT_LAKEHOUSE}.{BACKUP_TABLE} (

        pipeline_id         STRING      NOT NULL,

        pipeline_name       STRING,    

        definition_json     STRING,    
        backed_up_at        TIMESTAMP
) USING DELTA
""")
print(f"✅ Backup table ready: {CHECKPOINT_LAKEHOUSE}.{BACKUP_TABLE}")

## 4. Deploy Checkpoint Helper Notebook
Creates (or updates) the `_checkpoint_helper` notebook in the workspace. This lightweight notebook is called by pipelines to CHECK which activities are completed and UPDATE checkpoint records.

In [ ]:
# ── Build the helper notebook content (.ipynb format) ──
# NOTE: We build each cell's source as a list of lines to avoid triple-quote escaping issues.

helper_configure_lines = [
    '%%configure -f\n',
    '{\n',
    '    "defaultLakehouse": {\n',
    '        "name": {\n',
    '            "parameterName": "LH_Name",\n',
    '            "defaultValue": "publish"\n',
    '        },\n',
    '        "workspaceId": {\n',
    '            "parameterName": "WorkSpace_ID",\n',
    f'            "defaultValue": "{WORKSPACE_ID}"\n',
    '        }\n',
    '    }\n',
    '}',
]

helper_params_lines = [
    '# Parameters — set by the calling pipeline\n',
    'mode = "CHECK_ALL"  # CHECK_ALL or UPDATE or RESET\n',
    'checkpoint_lakehouse = ""\n',
    'checkpoint_table = "pipeline_activity_checkpoints"\n',
    'pipeline_name = ""\n',
    'pipeline_id = ""\n',
    'run_id = ""\n',
    'activity_name = ""\n',
    'activity_type = ""\n',
    'status = ""\n',
    'error_message = ""\n',
    'original_retry_count = "0"\n',
    'agent_workspace_id = ""\n',
    'agent_pipeline_id = ""\n',
    'source_workspace_id = ""\n',
    'source_workspace_name = ""',
]

helper_logic_lines = [
    'from pyspark.sql import SparkSession\n',
    'from datetime import datetime, timezone\n',
    '\n',
    'spark = SparkSession.builder.getOrCreate()\n',
    '\n',
    '# Guard against None parameters\n',
    'pipeline_name = pipeline_name or ""\n',
    'activity_name = activity_name or ""\n',
    'pipeline_id = pipeline_id or ""\n',
    'run_id = run_id or ""\n',
    'activity_type = activity_type or ""\n',
    'status = status or ""\n',
    'checkpoint_lakehouse = checkpoint_lakehouse or ""\n',
    'checkpoint_table = checkpoint_table or ""\n',
    'agent_workspace_id = agent_workspace_id or ""\n',
    'agent_pipeline_id = agent_pipeline_id or ""\n',
    'source_workspace_id = source_workspace_id or ""\n',
    'source_workspace_name = source_workspace_name or ""\n',
    '\n',
    'if mode == "CHECK_ALL":\n',
    '    try:\n',
    '        esc_pipeline = pipeline_name.replace("\'", "\'\'")\n',
    '        # Check if ANY checkpoint records exist for this pipeline\n',
    '        record_df = spark.sql(\n',
    '            f"SELECT COUNT(*) AS cnt FROM {checkpoint_lakehouse}.{checkpoint_table} "\n',
    '            f"WHERE pipeline_name = \'{esc_pipeline}\'"\n',
    '        )\n',
    '        total_records = record_df.collect()[0]["cnt"]\n',
    '        if total_records > 0:\n',
    '            # Records exist — return all COMPLETED activities so they get skipped\n',
    '            df = spark.sql(\n',
    '                f"SELECT DISTINCT activity_name FROM {checkpoint_lakehouse}.{checkpoint_table} "\n',
    '                f"WHERE pipeline_name = \'{esc_pipeline}\' AND status = \'COMPLETED\'"\n',
    '            )\n',
    '            completed_list = ",".join([r["activity_name"] for r in df.collect()])\n',
    '            failed_df = spark.sql(\n',
    '                f"SELECT COUNT(*) AS cnt FROM {checkpoint_lakehouse}.{checkpoint_table} "\n',
    '                f"WHERE pipeline_name = \'{esc_pipeline}\' AND status = \'FAILED\'"\n',
    '            )\n',
    '            failed_count = failed_df.collect()[0]["cnt"]\n',
    '            print(f"Checkpoint: {total_records} records, {len(completed_list.split(\',\')) if completed_list else 0} completed, {failed_count} failed")\n',
    '            print(f"Skipping: {completed_list}")\n',
    '        else:\n',
    '            # No records — fresh run\n',
    '            completed_list = ""\n',
    '            print("Checkpoint: no records found — running all activities")\n',
    '    except Exception as e:\n',
    '        print(f"Checkpoint read failed (running all activities): {e}")\n',
    '        completed_list = ""\n',
    '    mssparkutils.notebook.exit(completed_list)\n',
    '\n',
    'elif mode == "UPDATE":\n',
    '    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")\n',
    '    esc_pipeline = pipeline_name.replace("\'", "\'\'")\n',
    '    esc_name = activity_name.replace("\'", "\'\'")\n',
    '    esc_error = (error_message or "").replace("\'", "\'\'")[:500]\n',
    '    error_val = f"\'{esc_error}\'" if error_message else "NULL"\n',
    '    needs_rerun_val = "false" if status == "COMPLETED" else "true"\n',
    '    merge_sql = (\n',
    '        f"MERGE INTO {checkpoint_lakehouse}.{checkpoint_table} AS t "\n',
    '        f"USING (SELECT 1 AS _dummy) AS s "\n',
    '        f"ON t.pipeline_name = \'{esc_pipeline}\' AND t.activity_name = \'{esc_name}\' "\n',
    '        f"WHEN MATCHED THEN UPDATE SET "\n',
    '        f"t.status = \'{status}\', t.pipeline_id = \'{pipeline_id}\', "\n',
    '        f"t.run_id = \'{run_id}\', t.activity_type = \'{activity_type}\', "\n',
    '        f"t.error_message = {error_val}, t.needs_rerun = {needs_rerun_val}, "\n',
    '        f"t.original_retry_count = \'{original_retry_count}\', "\n',
    '        f"t.resolution_method = \'PIPELINE_CHECKPOINT\' "\n',
    '        f"WHEN NOT MATCHED THEN INSERT "\n',
    '        f"(pipeline_name, pipeline_id, run_id, activity_name, activity_type, "\n',
    '        f"status, error_message, needs_rerun, resolved_at, resolution_method, original_retry_count) "\n',
    '        f"VALUES (\'{esc_pipeline}\', \'{pipeline_id}\', \'{run_id}\', \'{esc_name}\', \'{activity_type}\', "\n',
    '        f"\'{status}\', {error_val}, {needs_rerun_val}, "\n',
    '        f"cast(\'{now}\' AS TIMESTAMP), \'PIPELINE_CHECKPOINT\', \'{original_retry_count}\')"\n',
    '    )\n',
    '    spark.sql(merge_sql)\n',
    '\n',
    '    # Notify AI Foundry Agent on failure\n',
    '    if status == "FAILED" and agent_workspace_id and agent_pipeline_id:\n',
    '        try:\n',
    '            import sempy.fabric as _fabric\n',
    '            import json as _json\n',
    '            _rc = _fabric.FabricRestClient()\n',
    '            _api = "https://api.fabric.microsoft.com/v1"\n',
    '            _job_body = {\n',
    '                "executionData": {\n',
    '                    "parameters": {\n',
    '                        "Type": "PipelineActivityFailure",\n',
    '                        "Source": "SelfHealingAgent",\n',
    '                        "WorkspaceId": source_workspace_id,\n',
    '                        "WorkspaceName": source_workspace_name,\n',
    '                        "JobInstanceId": run_id,\n',
    '                        "PipelineName": pipeline_name,\n',
    '                        "PipelineId": pipeline_id,\n',
    '                        "ActivityName": activity_name,\n',
    '                        "ActivityType": activity_type,\n',
    '                        "ErrorMessage": (error_message or "")[:500],\n',
    '                    }\n',
    '                }\n',
    '            }\n',
    '            _resp = _rc.post(\n',
    '                f"{_api}/workspaces/{agent_workspace_id}/items/{agent_pipeline_id}/jobs/instances?jobType=Pipeline",\n',
    '                json=_job_body,\n',
    '            )\n',
    '            if _resp.status_code in (200, 201, 202):\n',
    '                print(f"Agent notified: {pipeline_name}/{activity_name} -> PipelineMonitoring triggered")\n',
    '            else:\n',
    '                print(f"Agent notification failed: HTTP {_resp.status_code} - {_resp.text[:200]}")\n',
    '        except Exception as _e:\n',
    '            print(f"Agent notification error (non-blocking): {_e}")\n',
    '\n',
    '    mssparkutils.notebook.exit("OK")\n',
    '\n',
    'elif mode == "RESET":\n',
    '    try:\n',
    '        esc_pipeline = pipeline_name.replace("\'", "\'\'")\n',
    '        spark.sql(\n',
    '            f"DELETE FROM {checkpoint_lakehouse}.{checkpoint_table} "\n',
    '            f"WHERE pipeline_name = \'{esc_pipeline}\'"\n',
    '        )\n',
    '        print(f"Checkpoint reset: all records deleted for {pipeline_name}")\n',
    '        mssparkutils.notebook.exit("RESET_OK")\n',
    '    except Exception as e:\n',
    '        print(f"Checkpoint reset failed: {e}")\n',
    '        mssparkutils.notebook.exit(f"RESET_FAILED: {e}")\n',
    '\n',
    'else:\n',
    '    mssparkutils.notebook.exit("UNKNOWN_MODE")',
]

# Build .ipynb JSON
helper_ipynb = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernel_info": {"name": "synapse_pyspark"},
        "kernelspec": {"name": "synapse_pyspark", "display_name": "Synapse PySpark", "language": "Python"},
        "language_info": {"name": "python"},
    },
    "cells": [
        {
            "cell_type": "code",
            "metadata": {},
            "source": helper_configure_lines,
            "outputs": [],
            "execution_count": None,
        },
        {
            "cell_type": "code",
            "metadata": {"tags": ["parameters"]},
            "source": helper_params_lines,
            "outputs": [],
            "execution_count": None,
        },
        {
            "cell_type": "code",
            "metadata": {},
            "source": helper_logic_lines,
            "outputs": [],
            "execution_count": None,
        },
    ],
}

payload_b64 = base64.b64encode(json.dumps(helper_ipynb).encode("utf-8")).decode("utf-8")

# ── Check if helper notebook already exists ──
resp = rest_client.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items?type=Notebook")
resp.raise_for_status()
existing_notebooks = {nb["displayName"]: nb["id"] for nb in resp.json().get("value", [])}

helper_notebook_id = None

if HELPER_NOTEBOOK_NAME in existing_notebooks:
    helper_notebook_id = existing_notebooks[HELPER_NOTEBOOK_NAME]
    update_body = {
        "definition": {
            "format": "ipynb",
            "parts": [{"path": "notebook-content.ipynb", "payload": payload_b64, "payloadType": "InlineBase64"}],
        }
    }
    resp = rest_client.post(
        f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{helper_notebook_id}/updateDefinition",
        json=update_body,
    )
    resp = _wait_for_long_operation(resp, rest_client)
    print(f"✅ Helper notebook updated: {HELPER_NOTEBOOK_NAME} ({helper_notebook_id})")
else:
    create_body = {
        "displayName": HELPER_NOTEBOOK_NAME,
        "type": "Notebook",
        "definition": {
            "format": "ipynb",
            "parts": [{"path": "notebook-content.ipynb", "payload": payload_b64, "payloadType": "InlineBase64"}],
        },
    }
    resp = rest_client.post(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items", json=create_body)
    resp = _wait_for_long_operation(resp, rest_client)
    if resp.status_code in (200, 201):
        helper_notebook_id = resp.json().get("id")
        print(f"✅ Helper notebook created: {HELPER_NOTEBOOK_NAME} ({helper_notebook_id})")
    else:
        resp.raise_for_status()

# ── Resolve Agent Pipeline ID if not set ──
if NOTIFY_AGENT_ON_FAILURE and AGENT_WORKSPACE_ID and not AGENT_PIPELINE_ID:
    resp_ap = rest_client.get(f"{FABRIC_API}/workspaces/{AGENT_WORKSPACE_ID}/items?type=DataPipeline")
    if resp_ap.status_code == 200:
        for p in resp_ap.json().get("value", []):
            if p["displayName"] == AGENT_PIPELINE_NAME:
                AGENT_PIPELINE_ID = p["id"]
                break
    if AGENT_PIPELINE_ID:
        print(f"✅ Agent pipeline resolved: {AGENT_PIPELINE_NAME} ({AGENT_PIPELINE_ID})")
    else:
        print(f"⚠️  Agent pipeline '{AGENT_PIPELINE_NAME}' not found in workspace {AGENT_WORKSPACE_ID}")

# ── Resolve source workspace display name ──
_src_ws_name = WORKSPACE_ID
_src_resp = rest_client.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}")
if _src_resp.status_code == 200:
    _src_ws_name = _src_resp.json().get("displayName", WORKSPACE_ID)

print(f"   Helper notebook ID: {helper_notebook_id}")
print(f"   Agent pipeline ID:  {AGENT_PIPELINE_ID or '(not configured)'}")
assert helper_notebook_id, "Failed to create/find helper notebook"

## 5. Discover Pipelines
Lists all pipelines in the workspace and applies include/exclude filters.

In [ ]:
assert WORKSPACE_ID, "Set WORKSPACE_ID in the Parameters cell"

# ── List all pipelines ──
resp = rest_client.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items?type=DataPipeline")
resp.raise_for_status()
all_pipelines = resp.json().get("value", [])

print(f"Found {len(all_pipelines)} pipelines in workspace\n")


def _matches_filter(name, patterns):
    """Check if name matches any fnmatch pattern in the list."""
    return any(fnmatch.fnmatch(name, p) for p in patterns)


# ── Apply filters ──
filtered_pipelines = []
for p in all_pipelines:
    name = p["displayName"]

    # Exclude patterns
    if EXCLUDE_PIPELINES and _matches_filter(name, EXCLUDE_PIPELINES):
        print(f"  ✖  {name}  (excluded by filter)")
        continue

    # Include patterns (empty = include all)
    if INCLUDE_PIPELINES and not _matches_filter(name, INCLUDE_PIPELINES):
        print(f"  ✖  {name}  (not in include list)")
        continue

    filtered_pipelines.append(p)
    print(f"  ✔  {name}  ({p['id'][:12]}...)")

print(f"\n{len(filtered_pipelines)} pipelines selected for onboarding")

## 6. Pipeline Transformation Logic
The core function that injects checkpoint logic into a pipeline definition. For each active activity, it:
1. **Overrides retry → 0** so failures surface immediately for RCA (original count preserved)
2. Wraps it in an `IfCondition` — skip if already completed
3. Adds `_chk_upd_X` (on Success) — writes COMPLETED to checkpoint
4. Adds `_chk_fail_X` (on Failure) — writes FAILED + error details to checkpoint
5. Adds `_chk_re_fail_X` (Fail activity) — re-propagates the error so the IfCondition fails properly
6. Rewires all dependencies to point to the wrapper activities
7. **🔄 Recursive sub-pipeline resolution** — discovers `ExecutePipeline` activities and recursively transforms child pipelines (with cycle-guard to prevent infinite loops)
8. **💾 `clone_pipeline_with_backup()`** — clones the original pipeline as a Fabric backup item before any modification

In [ ]:
import re
import copy
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════════
# Utility: Clone pipeline with backup name before any modification
# ═══════════════════════════════════════════════════════════════════════════════
def clone_pipeline_with_backup(rest_client, workspace_id, pipeline_id, pipeline_name, pipeline_def_json, backup_suffix="_backup"):
    """
    Clone a pipeline as a Fabric backup item before modification.
    Returns (backup_item_id, backup_name) or raises on failure.
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"{pipeline_name}{backup_suffix}_{ts}"

    b64_payload = base64.b64encode(json.dumps(pipeline_def_json).encode("utf-8")).decode("utf-8")
    create_body = {
        "displayName": backup_name,
        "type": "DataPipeline",
        "definition": {
            "parts": [{
                "path": "pipeline-content.json",
                "payload": b64_payload,
                "payloadType": "InlineBase64",
            }]
        },
    }
    resp = rest_client.post(f"{FABRIC_API}/workspaces/{workspace_id}/items", json=create_body)
    resp = _wait_for_long_operation(resp, rest_client)
    if resp.status_code not in (200, 201):
        raise Exception(f"Backup pipeline creation failed: {resp.status_code} {resp.text[:300]}")
    backup_id = resp.json().get("id")
    print(f"    ✅ Backup pipeline created: {backup_name} ({backup_id})")
    return backup_id, backup_name


# ═══════════════════════════════════════════════════════════════════════════════
# Core: transform_pipeline — injects checkpoint logic into one pipeline
# ═══════════════════════════════════════════════════════════════════════════════
def transform_pipeline(pipeline_def, helper_notebook_id, pipeline_item_id,
                       pipeline_display_name=None,
                       rest_client=None, workspace_id=None, _visited=None):
    """
    Transform a pipeline definition to add checkpoint logic.

    Args:
        pipeline_def:         The pipeline-content.json dict
        helper_notebook_id:   GUID of the _checkpoint_helper notebook
        pipeline_item_id:     The pipeline Fabric item ID
        pipeline_display_name: The human-readable pipeline name (used in checkpoint table)
        rest_client:          (Optional) FabricRestClient — enables recursive sub-pipeline resolution
        workspace_id:         (Optional) Workspace GUID — required with rest_client
        _visited:             (Internal) Set of pipeline IDs already processed (cycle guard)

    Returns: (modified_def, stats_dict)
        stats_dict includes 'sub_transforms' — a list of dicts for each child pipeline that
        also needs onboarding: {id, name, modified_def, original_def, stats}
    """
    # ── Cycle guard for recursive sub-pipeline resolution ──
    if _visited is None:
        _visited = set()
    if pipeline_item_id in _visited:
        return pipeline_def, {"skipped": True, "reason": f"already visited (circular ref: {pipeline_item_id[:12]})"}
    _visited.add(pipeline_item_id)

    # Use literal display name if provided; fall back to @pipeline().Pipeline
    _pl_name_val = pipeline_display_name if pipeline_display_name else "@pipeline().Pipeline"

    sub_transforms = []  # Collect child pipeline transformations

    props = pipeline_def.get("properties", {})
    original_activities = props.get("activities", [])

    if not original_activities:
        return pipeline_def, {"skipped": True, "reason": "no activities"}

    # ── Auto-detect connection from existing TridentNotebook activities only ──
    def _find_notebook_connection(activities):
        """Recursively search for externalReferences.connection on TridentNotebook activities."""
        for act in activities:
            if act.get("type") == "TridentNotebook":
                conn = act.get("externalReferences", {}).get("connection")
                if conn:
                    return conn
            # Search inside control activities (IfCondition, ForEach, Switch, Until)
            tp = act.get("typeProperties", {})
            for branch_key in ("ifTrueActivities", "ifFalseActivities", "activities"):
                branch = tp.get(branch_key, [])
                if branch:
                    result = _find_notebook_connection(branch)
                    if result:
                        return result
            # Switch cases
            for case in tp.get("cases", []):
                result = _find_notebook_connection(case.get("activities", []))
                if result:
                    return result
            # Default activities for Switch
            default_acts = tp.get("defaultActivities", [])
            if default_acts:
                result = _find_notebook_connection(default_acts)
                if result:
                    return result
        return None

    notebook_connection = _find_notebook_connection(original_activities)

    def _make_notebook_activity(name, depends_on, policy, parameters, extra_props=None):
        """Build a TridentNotebook activity with connection info."""
        activity = {
            "name": name,
            "type": "TridentNotebook",
            "dependsOn": depends_on,
            "policy": policy,
            "typeProperties": {
                "notebookId": helper_notebook_id,
                "workspaceId": WORKSPACE_ID,
                "parameters": parameters,
            },
        }
        if notebook_connection:
            activity["externalReferences"] = {"connection": notebook_connection}
        if extra_props:
            activity.update(extra_props)
        return activity

    # ── Check if already onboarded → unwrap originals and re-onboard cleanly ──
    if any(a["name"] == "_chk_load" for a in original_activities):
        restored_activities = []
        for act in original_activities:
            if act["name"].startswith("_chk_if_") and act["type"] == "IfCondition":
                # Extract the original activity from ifTrueActivities
                true_acts = act.get("typeProperties", {}).get("ifTrueActivities", [])
                original_act = None
                original_retry = 0
                for inner in true_acts:
                    if not inner["name"].startswith("_chk_"):
                        original_act = copy.deepcopy(inner)
                    elif inner["name"].startswith("_chk_upd_"):
                        # Recover original retry count stored during onboarding
                        retry_str = inner.get("typeProperties", {}).get("parameters", {}).get(
                            "original_retry_count", {}
                        ).get("value", "0")
                        try:
                            original_retry = int(retry_str)
                        except (ValueError, TypeError):
                            original_retry = 0

                if original_act:
                    # Reconstruct dependencies from the IfCondition wrapper
                    wrapper_deps = act.get("dependsOn", [])
                    original_deps = []
                    for dep in wrapper_deps:
                        dep_name = dep["activity"]
                        if dep_name == "_chk_var":
                            pass  # Was an entry point — no original deps
                        elif dep_name.startswith("_chk_if_"):
                            original_deps.append({
                                "activity": dep_name[len("_chk_if_"):],
                                "dependencyConditions": dep.get("dependencyConditions", ["Succeeded"]),
                            })
                        else:
                            original_deps.append(copy.deepcopy(dep))

                    original_act["dependsOn"] = original_deps
                    if original_retry > 0:
                        if "policy" not in original_act:
                            original_act["policy"] = {}
                        original_act["policy"]["retry"] = original_retry
                    restored_activities.append(original_act)
            elif not act["name"].startswith("_chk_"):
                # Non-checkpoint activity (inactive, etc.) — keep as-is
                restored_activities.append(copy.deepcopy(act))

        original_activities = restored_activities
        if not original_activities:
            return pipeline_def, {"skipped": True, "reason": "only checkpoint activities found"}
        # Re-detect connection from the restored activities
        notebook_connection = _find_notebook_connection(original_activities)
        print(f"    ♻️  Re-onboarding: extracted {len(original_activities)} original activities, re-wrapping with latest checkpoint logic")

    # ── Classify activities ──
    active_names = set()
    inactive_names = set()
    for act in original_activities:
        is_inactive = (
            act.get("state", "").lower() == "inactive"
            or act.get("inactive", False) is True
            or act.get("policy", {}).get("state", "").lower() == "inactive"
        )
        if is_inactive:
            inactive_names.add(act["name"])
        else:
            active_names.add(act["name"])

    # ══════════════════════════════════════════════════════════════════════════
    # RECURSIVE SUB-PIPELINE RESOLUTION (deep scan)
    # ══════════════════════════════════════════════════════════════════════════

    def _find_all_execute_pipeline_refs(activities, depth=0):
        """
        Recursively walk through ALL activities (including those nested inside
        ForEach, IfCondition, Switch, Until, etc.) and return a list of
        (activity_name, pipeline_reference_id) for every ExecutePipeline found.
        """
        refs = []
        for act in activities:
            act_type = act.get("type", "")
            act_name = act.get("name", "?")

            # ── Check if this activity IS an ExecutePipeline ──
            if act_type == "ExecutePipeline":
                tp = act.get("typeProperties", {})
                sub_ref = (
                    tp.get("pipeline", {}).get("referenceName")
                    or tp.get("pipelineReference", {}).get("referenceName")
                    or tp.get("pipelineId")
                )
                if sub_ref:
                    refs.append((act_name, sub_ref, depth))

            # ── Recurse into nested activities of control activities ──
            tp = act.get("typeProperties", {})

            for nested_acts in [tp.get("activities", [])]:
                if nested_acts:
                    refs.extend(_find_all_execute_pipeline_refs(nested_acts, depth + 1))

            for branch_key in ("ifTrueActivities", "ifFalseActivities"):
                branch = tp.get(branch_key, [])
                if branch:
                    refs.extend(_find_all_execute_pipeline_refs(branch, depth + 1))

            for case in tp.get("cases", []):
                case_acts = case.get("activities", [])
                if case_acts:
                    refs.extend(_find_all_execute_pipeline_refs(case_acts, depth + 1))
            default_acts = tp.get("defaultActivities", [])
            if default_acts:
                refs.extend(_find_all_execute_pipeline_refs(default_acts, depth + 1))

        return refs

    if rest_client and workspace_id:
        all_exec_refs = _find_all_execute_pipeline_refs(original_activities)

        seen_refs = set()
        unique_refs = []
        for act_name, sub_ref, depth in all_exec_refs:
            if sub_ref not in seen_refs:
                seen_refs.add(sub_ref)
                unique_refs.append((act_name, sub_ref, depth))

        if unique_refs:
            indent = "    "
            print(f"{indent}🔍 Found {len(all_exec_refs)} ExecutePipeline reference(s) "
                  f"({len(unique_refs)} unique) across all nesting levels")

        for act_name, sub_ref, depth in unique_refs:
            if sub_ref in _visited:
                print(f"{indent}⏭️  '{act_name}' → {sub_ref[:12]}... (already visited, skipping cycle)")
                continue
            try:
                depth_indicator = "  " * depth + "↳ " if depth > 0 else ""

                # Fetch subpipeline item metadata (displayName)
                sub_item_resp = rest_client.get(
                    f"{FABRIC_API}/workspaces/{workspace_id}/items/{sub_ref}"
                )
                sub_name = sub_ref[:12]
                if sub_item_resp.status_code == 200:
                    sub_name = sub_item_resp.json().get("displayName", sub_ref[:12])

                # Fetch subpipeline definition
                sub_def_resp = rest_client.post(
                    f"{FABRIC_API}/workspaces/{workspace_id}/items/{sub_ref}/getDefinition"
                )
                sub_def_resp = _wait_for_long_operation(sub_def_resp, rest_client)

                if sub_def_resp.status_code != 200:
                    print(f"{indent}⚠️  {depth_indicator}Could not fetch subpipeline '{sub_name}' "
                          f"(called by '{act_name}'): HTTP {sub_def_resp.status_code}")
                    continue

                sub_pipeline_json = None
                for part in sub_def_resp.json().get("definition", {}).get("parts", []):
                    if part.get("path") == "pipeline-content.json":
                        sub_pipeline_json = json.loads(
                            base64.b64decode(part["payload"]).decode("utf-8")
                        )
                        break

                if not sub_pipeline_json:
                    print(f"{indent}⚠️  {depth_indicator}No pipeline-content.json in subpipeline '{sub_name}'")
                    continue

                # Recursively transform the sub-pipeline
                sub_modified, sub_stats = transform_pipeline(
                    sub_pipeline_json, helper_notebook_id, sub_ref,
                    pipeline_display_name=sub_name,
                    rest_client=rest_client, workspace_id=workspace_id,
                    _visited=_visited,
                )
                if not sub_stats.get("skipped"):
                    sub_transforms.append({
                        "id": sub_ref,
                        "name": sub_name,
                        "modified_def": sub_modified,
                        "original_def": sub_pipeline_json,
                        "stats": sub_stats,
                    })
                    print(f"{indent}🔄 {depth_indicator}Subpipeline '{sub_name}' queued for onboarding "
                          f"({sub_stats['active_wrapped']} activities, depth={depth})")
                else:
                    print(f"{indent}⏭️  {depth_indicator}Subpipeline '{sub_name}' skipped: "
                          f"{sub_stats.get('reason')}")

                # Collect any nested sub-transforms from deeper levels
                if sub_stats.get("sub_transforms"):
                    sub_transforms.extend(sub_stats["sub_transforms"])

            except Exception as e:
                print(f"{indent}⚠️  Error resolving subpipeline {sub_ref} "
                      f"(called by '{act_name}'): {e}")

    # ── Validate: no commas in activity names ──
    for name in active_names:
        assert "," not in name, f"Activity name contains comma (unsupported): '{name}'"

    # ── Add pipeline variable for checkpoint data ──
    variables = copy.deepcopy(props.get("variables", {}))
    variables["_completed_list"] = {"type": "String", "defaultValue": ""}

    new_activities = []
    retry_overrides = []
    skipped_dep_warnings = []

    # ── Activity types that Fabric does NOT allow inside IfCondition ──
    NON_NESTABLE_TYPES = {"IfCondition", "ForEach", "Switch", "Until", "ExecutePipeline"}

    # Track which activities got IfCondition wrapper vs direct placement
    wrapped_names = set()    # Activities wrapped in _chk_if_X
    unwrapped_names = set()  # Control-flow activities placed directly

    # ── 1. _chk_load: load completed activities from checkpoint table ──
    chk_load = _make_notebook_activity(
        name="_chk_load",
        depends_on=[],
        policy={"timeout": "0.00:10:00", "retry": 0, "retryIntervalInSeconds": 30},
        parameters={
            "mode": {"value": "CHECK_ALL", "type": "string"},
            "checkpoint_lakehouse": {"value": CHECKPOINT_LAKEHOUSE, "type": "string"},
            "checkpoint_table": {"value": CHECKPOINT_TABLE, "type": "string"},
            "pipeline_name": {"value": _pl_name_val, "type": "string"},
        },
    )
    new_activities.append(chk_load)

    # ── 2. _chk_var: store completed list in pipeline variable ──
    chk_var = {
        "name": "_chk_var",
        "type": "SetVariable",
        "dependsOn": [{"activity": "_chk_load", "dependencyConditions": ["Succeeded"]}],
        "typeProperties": {
            "variableName": "_completed_list",
            "value": {
                "value": "@string(activity('_chk_load').output.result.exitValue)",
                "type": "Expression",
            },
        },
    }
    new_activities.append(chk_var)

    # ── 3. For each activity, create checkpoint wrapper ──
    for act in original_activities:
        name = act["name"]

        # Keep deactivated activities as-is (Fabric already skips them)
        if name in inactive_names:
            new_activities.append(copy.deepcopy(act))
            continue

        # ── Remap dependencies ──
        original_deps = act.get("dependsOn", [])
        remapped_deps = []
        for dep in original_deps:
            dep_name = dep["activity"]
            if dep_name in active_names:
                if dep_name in wrapped_names:
                    remapped_deps.append({
                        "activity": f"_chk_if_{dep_name}",
                        "dependencyConditions": dep.get("dependencyConditions", ["Succeeded"]),
                    })
                elif dep_name in unwrapped_names:
                    remapped_deps.append({
                        "activity": f"_chk_upd_{dep_name}",
                        "dependencyConditions": dep.get("dependencyConditions", ["Succeeded"]),
                    })
                else:
                    # Not yet processed — forward reference; use _chk_if_ as default
                    remapped_deps.append({
                        "activity": f"_chk_if_{dep_name}",
                        "dependencyConditions": dep.get("dependencyConditions", ["Succeeded"]),
                    })
            else:
                remapped_deps.append(copy.deepcopy(dep))

        # Entry points (no deps) depend on _chk_var
        if not original_deps:
            remapped_deps = [{"activity": "_chk_var", "dependencyConditions": ["Succeeded"]}]
        elif all(dep["activity"] in inactive_names for dep in original_deps):
            remapped_deps = [{"activity": "_chk_var", "dependencyConditions": ["Succeeded"]}]

        # ── Override retry to 0 (checkpoint handles retries at pipeline level) ──
        original_retry = act.get("policy", {}).get("retry", 0)
        inner_act = copy.deepcopy(act)
        if "policy" not in inner_act:
            inner_act["policy"] = {}
        if original_retry > 0:
            retry_overrides.append({"activity": name, "original_retry": original_retry})
        inner_act["policy"]["retry"] = 0

        # ── Build success checkpoint ──
        chk_upd = _make_notebook_activity(
            name=f"_chk_upd_{name}",
            depends_on=[{"activity": name, "dependencyConditions": ["Succeeded"]}],
            policy={"timeout": "0.00:05:00", "retry": 0, "retryIntervalInSeconds": 30},
            parameters={
                "mode": {"value": "UPDATE", "type": "string"},
                "checkpoint_lakehouse": {"value": CHECKPOINT_LAKEHOUSE, "type": "string"},
                "checkpoint_table": {"value": CHECKPOINT_TABLE, "type": "string"},
                "pipeline_name": {"value": _pl_name_val, "type": "string"},
                "pipeline_id": {"value": pipeline_item_id, "type": "string"},
                "run_id": {"value": "@pipeline().RunId", "type": "string"},
                "activity_name": {"value": name, "type": "string"},
                "activity_type": {"value": act.get("type", ""), "type": "string"},
                "status": {"value": "COMPLETED", "type": "string"},
                "error_message": {"value": "", "type": "string"},
                "original_retry_count": {"value": str(original_retry), "type": "string"},
            },
        )

        # ── Build failure capture (+ agent notification params) ──
        chk_fail_params = {
            "mode": {"value": "UPDATE", "type": "string"},
            "checkpoint_lakehouse": {"value": CHECKPOINT_LAKEHOUSE, "type": "string"},
            "checkpoint_table": {"value": CHECKPOINT_TABLE, "type": "string"},
            "pipeline_name": {"value": _pl_name_val, "type": "string"},
            "pipeline_id": {"value": pipeline_item_id, "type": "string"},
            "run_id": {"value": "@pipeline().RunId", "type": "string"},
            "activity_name": {"value": name, "type": "string"},
            "activity_type": {"value": act.get("type", ""), "type": "string"},
            "status": {"value": "FAILED", "type": "string"},
            "error_message": {
                "value": f"@activity('{name}').error.message",
                "type": "string",
            },
            "original_retry_count": {"value": str(original_retry), "type": "string"},
        }
        # Add agent notification params if configured
        if NOTIFY_AGENT_ON_FAILURE and AGENT_PIPELINE_ID and AGENT_WORKSPACE_ID:
            chk_fail_params["agent_workspace_id"] = {"value": AGENT_WORKSPACE_ID, "type": "string"}
            chk_fail_params["agent_pipeline_id"] = {"value": AGENT_PIPELINE_ID, "type": "string"}
            chk_fail_params["source_workspace_id"] = {"value": WORKSPACE_ID, "type": "string"}
            chk_fail_params["source_workspace_name"] = {"value": _src_ws_name, "type": "string"}

        chk_fail = _make_notebook_activity(
            name=f"_chk_fail_{name}",
            depends_on=[{"activity": name, "dependencyConditions": ["Failed"]}],
            policy={"timeout": "0.00:05:00", "retry": 0, "retryIntervalInSeconds": 30},
            parameters=chk_fail_params,
        )

        # ── Re-propagate failure ──
        chk_re_fail = {
            "name": f"_chk_re_fail_{name}",
            "type": "Fail",
            "dependsOn": [{"activity": f"_chk_fail_{name}", "dependencyConditions": ["Succeeded"]}],
            "typeProperties": {
                "message": {
                    "value": f"@activity('{name}').error.message",
                    "type": "Expression",
                },
                "errorCode": "SELF_HEAL_RCA_REQUIRED",
            },
        }

        # ══════════════════════════════════════════════════════════════
        # Determine wrapping strategy based on activity type
        # ══════════════════════════════════════════════════════════════
        act_type = act.get("type", "")
        is_control_flow = act_type in NON_NESTABLE_TYPES

        if is_control_flow:
            # ── CONTROL-FLOW ACTIVITY — cannot be nested in IfCondition ──
            unwrapped_names.add(name)
            inner_act["dependsOn"] = remapped_deps
            new_activities.append(inner_act)
            new_activities.append(chk_upd)
            new_activities.append(chk_fail)
            new_activities.append(chk_re_fail)
        else:
            # ── STANDARD ACTIVITY — wrap in IfCondition for skip-on-rerun ──
            wrapped_names.add(name)
            inner_act["dependsOn"] = []

            if_condition_expr = (
                f"@not(contains(concat(',',variables('_completed_list'),','), ',{name},'))"
            )
            if_act = {
                "name": f"_chk_if_{name}",
                "type": "IfCondition",
                "dependsOn": remapped_deps,
                "typeProperties": {
                    "expression": {"value": if_condition_expr, "type": "Expression"},
                    "ifTrueActivities": [inner_act, chk_upd, chk_fail, chk_re_fail],
                    "ifFalseActivities": [],
                },
            }
            new_activities.append(if_act)

    # ── 4. _chk_reset: clear checkpoints after all activities succeed ──
    all_reset_deps = []
    for name in active_names:
        if name in wrapped_names:
            all_reset_deps.append({"activity": f"_chk_if_{name}", "dependencyConditions": ["Succeeded"]})
        elif name in unwrapped_names:
            all_reset_deps.append({"activity": f"_chk_upd_{name}", "dependencyConditions": ["Succeeded"]})

    chk_reset = _make_notebook_activity(
        name="_chk_reset",
        depends_on=all_reset_deps,
        policy={"timeout": "0.00:05:00", "retry": 0, "retryIntervalInSeconds": 30},
        parameters={
            "mode": {"value": "RESET", "type": "string"},
            "checkpoint_lakehouse": {"value": CHECKPOINT_LAKEHOUSE, "type": "string"},
            "checkpoint_table": {"value": CHECKPOINT_TABLE, "type": "string"},
            "pipeline_name": {"value": _pl_name_val, "type": "string"},
        },
    )
    new_activities.append(chk_reset)

    # ── Sanitize all retry policies (Fabric requires retryIntervalInSeconds >= 30) ──
    def _fix_retry_intervals(obj):
        """Recursively ensure retryIntervalInSeconds >= 30 everywhere in the definition."""
        if isinstance(obj, dict):
            if "retryIntervalInSeconds" in obj:
                val = obj["retryIntervalInSeconds"]
                if isinstance(val, (int, float)) and val < 30:
                    obj["retryIntervalInSeconds"] = 30
            for v in obj.values():
                _fix_retry_intervals(v)
        elif isinstance(obj, list):
            for item in obj:
                _fix_retry_intervals(item)

    # ── Build modified definition ──
    modified = copy.deepcopy(pipeline_def)
    modified["properties"]["activities"] = new_activities
    modified["properties"]["variables"] = variables

    _fix_retry_intervals(modified)

    stats = {
        "skipped": False,
        "original_count": len(original_activities),
        "active_wrapped": len(wrapped_names),
        "active_unwrapped": len(unwrapped_names),
        "inactive_preserved": len(inactive_names),
        "new_top_level_count": len(new_activities),
        "retry_overrides": retry_overrides,
        "skipped_dep_warnings": skipped_dep_warnings,
        "connection_found": notebook_connection is not None,
        "connection_value": notebook_connection,
        "sub_transforms": sub_transforms,
        "sub_pipeline_count": len(sub_transforms),
        "unwrapped_activities": list(unwrapped_names),
    }

    return modified, stats


print("✅ transform_pipeline() defined  (with recursive sub-pipeline support & control-flow handling)")
print("   ✅ clone_pipeline_with_backup() defined")

## 7. Preview Changes (Dry Run)
Shows what each pipeline will look like after onboarding — without applying any changes. Review this before setting `DRY_RUN = False`.

In [ ]:
# ── Helper: find all referenceName values in a nested structure ──
import re as _re
_guid_pattern = _re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', _re.IGNORECASE)

def _find_references(obj, path=""):
    """Recursively find all referenceName and notebookId values and their paths."""
    refs = []
    if isinstance(obj, dict):
        if "referenceName" in obj:
            refs.append((path + ".referenceName", obj["referenceName"], obj.get("type", "?")))
        if "notebookId" in obj:
            refs.append((path + ".notebookId", obj["notebookId"], "notebookId"))
        for k, v in obj.items():
            refs.extend(_find_references(v, f"{path}.{k}"))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            refs.extend(_find_references(item, f"{path}[{i}]"))
    return refs

# ── Build workspace item lookup ──
resp_items = rest_client.get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items")
resp_items.raise_for_status()
all_items = {item["id"]: item["displayName"] for item in resp_items.json().get("value", [])}
print(f"Workspace has {len(all_items)} items\n")

# ── Fetch definitions and preview transformations ──
preview_results = []

for p in filtered_pipelines:
    pid = p["id"]
    pname = p["displayName"]
    print(f"\n{'─' * 60}")
    print(f"  Pipeline: {pname}  ({pid[:12]}...)")
    print(f"{'─' * 60}")

    # Get definition
    resp = rest_client.post(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{pid}/getDefinition")
    resp = _wait_for_long_operation(resp, rest_client)

    if resp.status_code != 200:
        err_detail = ""
        try:
            err_detail = resp.json().get("error", {}).get("code", resp.text[:200])
        except Exception:
            err_detail = resp.text[:200]
        print(f"  ❌ Failed to get definition: HTTP {resp.status_code} — {err_detail}")
        preview_results.append({"name": pname, "id": pid, "status": "FETCH_FAILED", "error": err_detail})
        continue

    pipeline_json = None
    for part in resp.json().get("definition", {}).get("parts", []):
        if part.get("path") == "pipeline-content.json":
            pipeline_json = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
            break

    if not pipeline_json:
        print(f"  ❌ No pipeline-content.json found")
        preview_results.append({"name": pname, "id": pid, "status": "NO_CONTENT"})
        continue

    # ── DEBUG: Show top-level keys and all references ──
    print(f"\n  📋 Top-level keys: {list(pipeline_json.keys())}")
    # ── DEBUG: Dump TridentNotebook activities (typeProperties + externalReferences) ──
    for act in pipeline_json.get('properties', {}).get('activities', []):
        if act.get('type') == 'TridentNotebook':
            print(f"\n  🔬 TridentNotebook '{act['name']}':")
            print(f"    typeProperties: {json.dumps(act.get('typeProperties', {}), indent=4)[:500]}")
            ext_refs = act.get('externalReferences', {})
            if ext_refs:
                print(f"    externalReferences: {json.dumps(ext_refs)}")
            else:
                print(f"    ⚠️  No externalReferences found")
    # ── DEBUG: Show connections on ALL activities ──
    for act in pipeline_json.get('properties', {}).get('activities', []):
        ext_refs = act.get('externalReferences', {})
        if ext_refs:
            print(f"  🔌 {act['name']} [{act['type']}] connection: {json.dumps(ext_refs)}")

    orig_refs = _find_references(pipeline_json)
    if orig_refs:
        print(f"\n  🔗 References in ORIGINAL definition:")
        for path, val, ref_type in orig_refs:
            is_guid = bool(_guid_pattern.match(val))
            resolved = all_items.get(val, "NOT FOUND") if is_guid else "(not a GUID)"
            print(f"    {path}: {val} [{ref_type}] → {resolved}")

    # Transform (recursive — walks into ExecutePipeline sub-pipelines)
    modified_def, stats = transform_pipeline(
        pipeline_json, helper_notebook_id, pid,
        pipeline_display_name=pname,
        rest_client=rest_client, workspace_id=WORKSPACE_ID,
    )

    # ── DEBUG: Show all references in MODIFIED definition ──
    mod_refs = _find_references(modified_def)
    if mod_refs:
        print(f"\n  🔗 References in MODIFIED definition:")
        for path, val, ref_type in mod_refs:
            is_guid = bool(_guid_pattern.match(val))
            resolved = all_items.get(val, "NOT FOUND") if is_guid else "(not a GUID)"
            print(f"    {path}: {val} [{ref_type}] → {resolved}")

    if stats.get("skipped"):
        print(f"  ⏭️  Skipped: {stats.get('reason')}")
        preview_results.append({"name": pname, "id": pid, "status": "SKIPPED", "reason": stats["reason"]})
        continue

    # Show summary
    original_acts = pipeline_json.get("properties", {}).get("activities", [])
    print(f"\n  Original activities ({stats['original_count']}):")
    for act in original_acts:
        inactive = act.get("state", "").lower() == "inactive"
        tag = "  ⛔ DEACTIVATED" if inactive else ""
        deps = [d["activity"] for d in act.get("dependsOn", [])]
        dep_str = f"  (after: {deps})" if deps else ""
        print(f"    • {act['name']}  [{act['type']}]{dep_str}{tag}")

    print(f"\n  After onboarding ({stats['new_top_level_count']} top-level activities):")
    print(f"    ✅ {stats['active_wrapped']} activities wrapped with checkpoint IfCondition")
    if stats.get("active_unwrapped", 0) > 0:
        print(f"    🔧 {stats['active_unwrapped']} control-flow activities with direct checkpoint (no IfCondition wrapper):")
        for uw_name in stats.get("unwrapped_activities", []):
            print(f"       • {uw_name}")
    print(f"    ⛔ {stats['inactive_preserved']} deactivated activities preserved as-is")
    print(f"    📋 Added: _chk_load (notebook) + _chk_var (SetVariable)")
    print(f"    📋 Per activity: _chk_if_X (IfCondition) + _chk_upd_X + _chk_fail_X + _chk_re_fail_X")
    if stats.get("connection_found"):
        print(f"    🔌 Connection: {stats['connection_value']}")
    else:
        print(f"    ⚠️  No TridentNotebook connection found in original pipeline")
        print(f"       Checkpoint notebook activities will have no connection — may need manual setup")
    if stats.get("retry_overrides"):
        print(f"    ⚠️  Retry → 0 on {len(stats['retry_overrides'])} activities:")
        for r_info in stats["retry_overrides"]:
            print(f"       • {r_info}")
    if stats.get("skipped_dep_warnings"):
        print(f"    ⚠️  'Skipped' dependency conditions detected (review manually):")
        for w in stats["skipped_dep_warnings"]:
            print(f"       • {w}")
    if stats.get("sub_pipeline_count", 0) > 0:
        print(f"    🔄 {stats['sub_pipeline_count']} sub-pipeline(s) queued for onboarding:")
        for sub in stats["sub_transforms"]:
            sub_s = sub["stats"]
            print(f"       • {sub['name']} ({sub['id'][:12]}...) — {sub_s['active_wrapped']} activities")
            if sub_s.get("sub_pipeline_count", 0) > 0:
                for nested in sub_s["sub_transforms"]:
                    print(f"         ↳ {nested['name']} ({nested['id'][:12]}...) — {nested['stats']['active_wrapped']} activities")

    preview_results.append({
        "name": pname, "id": pid, "status": "READY",
        "original_count": stats["original_count"],
        "wrapped": stats["active_wrapped"],
        "inactive": stats["inactive_preserved"],
        "new_count": stats["new_top_level_count"],
        "modified_def": modified_def,
        "original_def": pipeline_json,
        "sub_transforms": stats.get("sub_transforms", []),
    })

# ── Summary ──
ready = [r for r in preview_results if r["status"] == "READY"]
skipped = [r for r in preview_results if r["status"] == "SKIPPED"]
failed = [r for r in preview_results if r["status"] not in ("READY", "SKIPPED")]

total_subs = sum(len(r.get("sub_transforms", [])) for r in ready)

print(f"\n{'=' * 60}")
print(f"  PREVIEW SUMMARY")
print(f"{'=' * 60}")
print(f"  Ready to onboard:  {len(ready)} parent pipeline(s) + {total_subs} sub-pipeline(s)")
print(f"  Skipped:           {len(skipped)}")
print(f"  Failed:            {len(failed)}")
if failed:
    for f in failed:
        print(f"    ❌ {f['name']}: {f.get('error', 'unknown')}")
if DRY_RUN:
    print(f"\n  ⚠️  DRY_RUN = True — no changes will be applied")
    print(f"  Set DRY_RUN = False and run the next cell to apply")
print(f"{'=' * 60}")

## 8. Apply Changes
Backs up original pipeline definitions to the backup table, then updates each pipeline with the checkpoint-wrapped version. **Only runs when `DRY_RUN = False`.**

In [ ]:
if DRY_RUN:
    print("⚠️  DRY_RUN = True — skipping. Set DRY_RUN = False to apply changes.")
else:
    ready_pipelines = [r for r in preview_results if r["status"] == "READY"]

    if not ready_pipelines:
        print("No pipelines ready for onboarding.")
    else:
        # ── Collect ALL pipelines to onboard (parents + discovered sub-pipelines) ──
        all_to_onboard = []
        for r in ready_pipelines:
            all_to_onboard.append({
                "id": r["id"], "name": r["name"],
                "modified_def": r["modified_def"], "original_def": r["original_def"],
                "wrapped": r.get("wrapped", 0), "is_sub": False,
            })
            # Add sub-pipelines discovered during recursive resolution
            for sub in r.get("sub_transforms", []):
                # Avoid duplicates (sub might appear under multiple parents)
                if not any(o["id"] == sub["id"] for o in all_to_onboard):
                    all_to_onboard.append({
                        "id": sub["id"], "name": sub["name"],
                        "modified_def": sub["modified_def"], "original_def": sub["original_def"],
                        "wrapped": sub["stats"].get("active_wrapped", 0), "is_sub": True,
                    })

        print(f"Onboarding {len(all_to_onboard)} pipelines "
              f"({len(ready_pipelines)} parent + {len(all_to_onboard) - len(ready_pipelines)} sub-pipelines)...\n")
        apply_results = []

        for r in all_to_onboard:
            pid = r["id"]
            pname = r["name"]
            sub_tag = " [SUB-PIPELINE]" if r["is_sub"] else ""
            print(f"[{pname}]{sub_tag}")

            # ── 1a. Clone pipeline as Fabric backup item ──
            try:
                backup_id, backup_name = clone_pipeline_with_backup(
                    rest_client, WORKSPACE_ID, pid, pname, r["original_def"]
                )
            except Exception as e:
                print(f"  ⚠️  Fabric backup clone failed (continuing with table backup): {e}")
                backup_id, backup_name = None, None

            # ── 1b. Backup original definition to Delta table ──
            backup_json = json.dumps(r["original_def"])
            now_utc = datetime.now(timezone.utc)
            backup_df = spark.createDataFrame([{
                "pipeline_id": pid,
                "pipeline_name": pname,
                "definition_json": backup_json,
                "backed_up_at": now_utc,
            }])

            backup_table_path = f"{CHECKPOINT_LAKEHOUSE}.{BACKUP_TABLE}"
            try:
                dt = DeltaTable.forName(spark, backup_table_path)
                dt.alias("t").merge(
                    backup_df.alias("s"),
                    "t.pipeline_id = s.pipeline_id"
                ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
            except Exception:
                backup_df.write.format("delta").mode("append").saveAsTable(backup_table_path)

            print(f"  ✅ Backup saved to {backup_table_path}")
            if backup_name:
                print(f"  ✅ Fabric clone: {backup_name}")

            # ── 2. Update pipeline definition via REST API ──
            modified_json = r["modified_def"]
            b64_payload = base64.b64encode(
                json.dumps(modified_json).encode("utf-8")
            ).decode("utf-8")

            update_body = {
                "definition": {
                    "parts": [{
                        "path": "pipeline-content.json",
                        "payload": b64_payload,
                        "payloadType": "InlineBase64",
                    }]
                }
            }

            resp = rest_client.post(
                f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{pid}/updateDefinition",
                json=update_body,
            )
            resp = _wait_for_long_operation(resp, rest_client)

            if resp.status_code in (200, 204):
                print(f"  ✅ Pipeline updated ({r['wrapped']} activities wrapped)")
                apply_results.append({"name": pname, "id": pid, "status": "SUCCESS", "is_sub": r["is_sub"]})
            else:
                print(f"  ❌ Update failed: HTTP {resp.status_code} — {resp.text[:200]}")
                apply_results.append({"name": pname, "id": pid, "status": "FAILED", "error": resp.text[:200], "is_sub": r["is_sub"]})

        # ── Summary ──
        succeeded = [r for r in apply_results if r["status"] == "SUCCESS"]
        failed_apply = [r for r in apply_results if r["status"] == "FAILED"]
        sub_succeeded = [r for r in succeeded if r.get("is_sub")]

        print(f"\n{'=' * 60}")
        print(f"  ONBOARDING COMPLETE")
        print(f"{'=' * 60}")
        print(f"  ✅ Succeeded: {len(succeeded)}  (incl. {len(sub_succeeded)} sub-pipelines)")
        print(f"  ❌ Failed:    {len(failed_apply)}")
        for r in succeeded:
            tag = " [SUB]" if r.get("is_sub") else ""
            print(f"    ✅ {r['name']}{tag}")
        for r in failed_apply:
            tag = " [SUB]" if r.get("is_sub") else ""
            print(f"    ❌ {r['name']}{tag}: {r.get('error', '')[:80]}")
        print(f"{'=' * 60}")

## 9. Verify Onboarding
Checks each onboarded pipeline to confirm the checkpoint wrapper activities are present.

In [ ]:
print("Verifying onboarded pipelines...\n")

for r in [r for r in preview_results if r["status"] == "READY"]:
    pid = r["id"]
    pname = r["name"]

    resp = rest_client.post(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{pid}/getDefinition")
    resp = _wait_for_long_operation(resp, rest_client)

    if resp.status_code != 200:
        print(f"  ❌ {pname}: could not fetch definition")
        continue

    pipeline_json = None
    for part in resp.json().get("definition", {}).get("parts", []):
        if part.get("path") == "pipeline-content.json":
            pipeline_json = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
            break

    if not pipeline_json:
        print(f"  ❌ {pname}: no pipeline-content.json")
        continue

    activities = pipeline_json.get("properties", {}).get("activities", [])
    act_names = {a["name"] for a in activities}
    has_load = "_chk_load" in act_names
    has_var = "_chk_var" in act_names
    if_count = sum(1 for n in act_names if n.startswith("_chk_if_"))
    upd_count = sum(1 for n in act_names if n.startswith("_chk_upd_"))
    fail_count = sum(1 for n in act_names if n.startswith("_chk_fail_"))
    re_fail_count = sum(1 for n in act_names if n.startswith("_chk_re_fail_"))

    has_variable = "_completed_list" in pipeline_json.get("properties", {}).get("variables", {})

    if has_load and has_var and if_count > 0 and has_variable:
        print(f"  ✅ {pname}: _chk_load ✓  _chk_var ✓  {if_count} IfConditions ✓  {upd_count} upd ✓  {fail_count} fail-capture ✓  {re_fail_count} re-fail ✓  variable ✓")
    else:
        issues = []
        if not has_load: issues.append("missing _chk_load")
        if not has_var: issues.append("missing _chk_var")
        if if_count == 0: issues.append("no _chk_if_ activities")

        if not has_variable: issues.append("missing _completed_list variable")        
        print(f"  ⚠️  {pname}: {', '.join(issues)}")

## 10. Monitor Checkpoints
Query the checkpoint table to see which activities have been completed across pipeline runs.

In [ ]:
checkpoint_df = spark.sql(f"""
    SELECT
        pipeline_name,
        activity_name,
        activity_type,
        status,
        needs_rerun,
        original_retry_count,
        failure_category,
        rca_notes,
        run_id,
        resolution_method,
        resolved_at
    FROM {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE}
    ORDER BY pipeline_name, resolved_at DESC
""")
total = checkpoint_df.count()
completed = checkpoint_df.filter("status = 'COMPLETED'").count()
failed = checkpoint_df.filter("status = 'FAILED'").count()
pipelines = checkpoint_df.select("pipeline_name").distinct().count()

print(f"Checkpoint records: {total}")
print(f"  Completed: {completed}")
print(f"  Failed:    {failed}")
print(f"  Pipelines: {pipelines}")
print()
checkpoint_df.show(100, truncate=50)

## 11. Rollback
Restores original pipeline definitions from the backup table. Use this if onboarding causes issues.

In [ ]:
# ⚠️  Uncomment and run to rollback specific pipelines or all pipelines

# ── Option A: Rollback a specific pipeline by name ──
# ROLLBACK_PIPELINE_NAME = "My_Pipeline"
#
# backup_row = spark.sql(f"""
#     SELECT pipeline_id, pipeline_name, definition_json
#     FROM {CHECKPOINT_LAKEHOUSE}.{BACKUP_TABLE}
#     WHERE pipeline_name = '{ROLLBACK_PIPELINE_NAME}'
#     ORDER BY backed_up_at DESC
#     LIMIT 1
# """).collect()
#
# if backup_row:
#     row = backup_row[0]
#     b64 = base64.b64encode(row["definition_json"].encode("utf-8")).decode("utf-8")
#     body = {"definition": {"parts": [{"path": "pipeline-content.json", "payload": b64, "payloadType": "InlineBase64"}]}}
#     resp = rest_client.post(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{row['pipeline_id']}/updateDefinition", json=body)
#     resp = _wait_for_long_operation(resp, rest_client)
#     if resp.status_code in (200, 204):
#         print(f"✅ Rolled back: {row['pipeline_name']}")
#     else:
#         print(f"❌ Rollback failed: HTTP {resp.status_code}")
# else:
#     print(f"No backup found for '{ROLLBACK_PIPELINE_NAME}'")

# ── Option B: Rollback ALL backed-up pipelines ──
# all_backups = spark.sql(f"""
#     SELECT pipeline_id, pipeline_name, definition_json
#     FROM {CHECKPOINT_LAKEHOUSE}.{BACKUP_TABLE}
# """).collect()
#
# for row in all_backups:
#     b64 = base64.b64encode(row["definition_json"].encode("utf-8")).decode("utf-8")
#     body = {"definition": {"parts": [{"path": "pipeline-content.json", "payload": b64, "payloadType": "InlineBase64"}]}}
#     resp = rest_client.post(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{row['pipeline_id']}/updateDefinition", json=body)
#     resp = _wait_for_long_operation(resp, rest_client)
#     status = "✅" if resp.status_code in (200, 204) else "❌"
#     print(f"  {status} {row['pipeline_name']}")

## 12. Reset Checkpoints
Clear checkpoint data for specific pipelines to force a full re-run on next execution.

In [ ]:
# ⚠️  Uncomment the section you want to run

# ── Option A: Clear checkpoints for a specific pipeline ──
# RESET_PIPELINE_NAME = "My_Pipeline"
# spark.sql(f"""
#     DELETE FROM {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE}
#     WHERE pipeline_name = '{RESET_PIPELINE_NAME}'
# """)
# print(f"✅ Checkpoints cleared for: {RESET_PIPELINE_NAME}")

# ── Option B: Clear ALL checkpoints (full reset) ──
# spark.sql(f"DELETE FROM {CHECKPOINT_LAKEHOUSE}.{CHECKPOINT_TABLE} WHERE 1=1")
# print("✅ All checkpoint records cleared")